# 03 LLM test with the correct chunks (no retrieval)

Feeds the LLM the **gold chunks** instead of retrieved ones. This separates "the LLM cannot answer or cite" from "retrieval missed it".

## How to use (one LLM per teammate, run in parallel)
1. Set `LLM_NAME` below: `qwen3b`, `qwen7b` or `aya8b` (aya needs the HF token and license).
2. Run all cells top to bottom on a GPU.
3. Send the zip to whoever combines the results. When all three summary files are in `results/`, run the last cell.

**What each LLM sees** (always 5 chunks, like the real pipeline):
- Answerable question: its gold chunks, plus random other chunks to reach 5, in shuffled order (seed 42).
- Not answerable: 5 random chunks. A correct LLM refuses.
- Set `PAD_WITH_DISTRACTORS = False` to give answerable questions only their gold chunks.

## 1. Choose the LLM

In [ ]:
# ============ CHANGE ONLY THIS LINE ============
LLM_NAME = "qwen3b"   # qwen3b | qwen7b | aya8b
# ===============================================

PAD_WITH_DISTRACTORS = True   # pad gold chunks with random chunks up to 5
N_CHUNKS = 5
SEED = 42

## 2. Setup

In [ ]:
import os, sys, json, time, random, subprocess
from pathlib import Path

REPO_URL = "https://github.com/yousefJooX/egyptian-personal-status-rag.git"
REPO_NAME = "egyptian-personal-status-rag"
IS_COLAB = "google.colab" in sys.modules
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

cwd = Path.cwd()
if (cwd / "run.py").exists() or (cwd / "demo").exists():
    ROOT = cwd
else:
    ROOT = cwd / REPO_NAME
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(ROOT)], check=True)

DEMO = ROOT / "demo" if (ROOT / "demo").exists() else ROOT
os.chdir(DEMO)   # blocks cache indexes in data/index relative to here
for p in (str(DEMO), str(ROOT)):
    if p not in sys.path:
        sys.path.insert(0, p)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "accelerate", "bitsandbytes", "sentence-transformers", "pandas", "tqdm"], check=True)
print("Environment:", "Colab" if IS_COLAB else "Kaggle" if IS_KAGGLE else "Local")
print("ROOT:", ROOT, "| working dir:", Path.cwd())

In [ ]:
token = os.environ.get("HF_TOKEN")
if not token:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        pass
if token:
    os.environ["HF_TOKEN"] = token
print("HF_TOKEN:", "loaded" if token else "not found (fine for qwen, required for aya8b)")

## 3. Load data and build the chunk lists

In [ ]:
import numpy as np
import pandas as pd

def find_file(name):
    for p in (DEMO / "data" / name, ROOT / "data" / name):
        if p.exists():
            return p
    raise FileNotFoundError(f"{name} not found in {DEMO/'data'} or {ROOT/'data'}")

def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def save_json(path, data):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

KS = (1, 3, 5, 10, 20)

def qgroup(item):
    """Question groups used in the breakdown. Adds item['qtype'] if the test set has it."""
    g = {"size": "multi-chunk" if len(item["gold_chunks"]) > 1 else "single-chunk"}
    qt = item.get("qtype") or item.get("type")   # the test set labels kinds as "type"
    if qt:
        g["qtype"] = qt
    return g

def score(per_q, T):
    """per_q: list of {qid, top (ranked chunk ids, up to 20)}. Answerable questions only (Task 7)."""
    ans = [p for p in per_q if T[p["qid"]]["answerable"]]
    out = {"n_answerable": len(ans)}
    for k in KS:
        rec, hit = [], 0
        for p in ans:
            gold = set(T[p["qid"]]["gold_chunks"])
            found = gold & set(p["top"][:k])
            rec.append(len(found) / len(gold))
            hit += bool(found)
        out[f"recall@{k}"] = float(np.mean(rec)) if rec else float("nan")
        out[f"hit@{k}"] = hit          # questions with at least one gold chunk in the top k
    for k in (5, 20):
        rr = []
        for p in ans:
            gold = set(T[p["qid"]]["gold_chunks"])
            ranks = [i for i, c in enumerate(p["top"][:k], 1) if c in gold]
            rr.append(1 / ranks[0] if ranks else 0.0)
        out[f"mrr@{k}"] = float(np.mean(rr)) if rr else float("nan")
    return out

import torch
from tqdm.auto import tqdm

if LLM_NAME not in ("qwen3b", "qwen7b", "aya8b"):
    raise ValueError("LLM_NAME must be qwen3b, qwen7b or aya8b")

chunks = load_json(find_file("chunks.json"))
test_set = load_json(find_file("test_set.json"))
chunks_by_id = {c["id"]: c for c in chunks}
T = {q["qid"]: q for q in test_set}
assert len(T) == len(test_set), "duplicate qid in test_set.json"
for q in test_set:
    missing = [g for g in q["gold_chunks"] if g not in chunks_by_id]
    assert not missing, f"{q['qid']}: gold chunk ids not in chunks.json: {missing}"

# fixed input chunks per question: identical for every LLM (seed 42)
rng = random.Random(SEED)
all_ids = [c["id"] for c in chunks]
INPUT = {}
for q in test_set:
    gold = list(q["gold_chunks"])
    if q["answerable"] and not PAD_WITH_DISTRACTORS:
        ids = gold
    else:
        pool = [i for i in all_ids if i not in gold]
        ids = gold + rng.sample(pool, max(0, N_CHUNKS - len(gold)))
        rng.shuffle(ids)
    INPUT[q["qid"]] = ids

n_ans = sum(q["answerable"] for q in test_set)
print(f"LLM: {LLM_NAME} | Chunks: {len(chunks)} | Questions: {len(test_set)} ({n_ans} answerable, {len(test_set)-n_ans} not)")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU will be very slow)")

## 4. Load the LLM (only this one)

In [ ]:
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
from blocks.llms import LLM
llm = LLM(LLM_NAME)
print("Loaded", LLM_NAME)

## 5. Run all questions

In [ ]:
answers = []
for item in tqdm(test_set, desc=LLM_NAME):
    top_chunks = [chunks_by_id[i] for i in INPUT[item["qid"]]]
    t0 = time.time()
    out = llm.generate(item["question"], top_chunks)
    sec = time.time() - t0
    answers.append({"qid": item["qid"], "answer": out["answer"], "cited_chunks": out["cited_chunks"],
                    "refused": out["refused"], "seconds": round(sec, 4), "input_chunks": INPUT[item["qid"]]})
peak_gpu_mb = int(torch.cuda.max_memory_allocated() // (1024 * 1024)) if torch.cuda.is_available() else 0
print(f"Done. {len(answers)} answers | peak GPU {peak_gpu_mb} MB")

## 6. Metrics
- **citation_acc**: answerable and not refused; correct if `cited_chunks` is non-empty and every cited chunk is gold (Task 7).
- **cite_all_gold**: same pool; cited chunks cover **all** gold chunks (matters for multi-chunk questions).
- **refusal_acc**: all questions. **false_refusal**: answerable questions the LLM refused. **correct_refusal**: unanswerable questions it refused.

In [ ]:
A = {a["qid"]: a for a in answers}

def llm_metrics(qids):
    ans = [q for q in qids if T[q]["answerable"]]
    un = [q for q in qids if not T[q]["answerable"]]
    pool = [q for q in ans if not A[q]["refused"]]
    ok = lambda q: bool(A[q]["cited_chunks"]) and set(A[q]["cited_chunks"]) <= set(T[q]["gold_chunks"])
    full = lambda q: set(T[q]["gold_chunks"]) <= set(A[q]["cited_chunks"])
    n_ref = sum(A[q]["refused"] == (not T[q]["answerable"]) for q in qids)
    return {
        "citation_acc": f"{sum(map(ok, pool))}/{len(pool)}",
        "cite_all_gold": f"{sum(map(full, pool))}/{len(pool)}",
        "refusal_acc": f"{n_ref}/{len(qids)}",
        "false_refusal": f"{sum(A[q]['refused'] for q in ans)}/{len(ans)}",
        "correct_refusal": f"{sum(A[q]['refused'] for q in un)}/{len(un)}",
        "_citation_acc": sum(map(ok, pool)) / len(pool) if pool else float("nan"),
        "_refusal_acc": n_ref / len(qids),
    }

allm = llm_metrics(list(T))
secs = [a["seconds"] for a in answers]
summary = {"llm": LLM_NAME, "pad_with_distractors": PAD_WITH_DISTRACTORS, "n_questions": len(T),
           **allm, "avg_seconds": round(float(np.mean(secs)), 3), "median_seconds": round(float(np.median(secs)), 3),
           "peak_gpu_mb": peak_gpu_mb}
display(pd.DataFrame([{k: v for k, v in summary.items() if not k.startswith("_")}]).set_index("llm"))

rows = []
for g in ("single-chunk", "multi-chunk"):
    qs = [q for q in T if T[q]["answerable"] and ("multi-chunk" if len(T[q]["gold_chunks"]) > 1 else "single-chunk") == g]
    if qs:
        m = llm_metrics(qs)
        rows.append({"group": g, "n": len(qs), "citation_acc": m["citation_acc"], "cite_all_gold": m["cite_all_gold"],
                     "false_refusal": m["false_refusal"]})
display(pd.DataFrame(rows))

## 7. Read 5 answers (required by Task 6)
For each, note whether the model follows the citation rule (article number after each fact). A 3B model may not.

In [ ]:
for q in random.Random(SEED).sample([k for k in T if T[k]["answerable"]], min(5, n_ans)):
    a = A[q]
    print(f"--- {q} | refused={a['refused']} | cited={a['cited_chunks']} | gold={T[q]['gold_chunks']}")
    print("Q:", T[q]["question"])
    print("A:", a["answer"])

## 8. Save files

In [ ]:
import shutil
results_dir = DEMO / "results"
results_dir.mkdir(exist_ok=True)
pre = f"llm_gold_{LLM_NAME}"
save_json(results_dir / f"{pre}_answers.json", answers)
save_json(results_dir / f"{pre}_summary.json", {**{k: v for k, v in summary.items() if not k.startswith("_")},
          "citation_acc_value": round(allm["_citation_acc"], 4), "refusal_acc_value": round(allm["_refusal_acc"], 4)})

# manual check: 20 answerable questions, seed 42; the team fills correct / faithful with 1 or 0
ids = sorted(q for q in T if T[q]["answerable"])
picked = random.Random(SEED).sample(ids, min(20, len(ids)))
pd.DataFrame([{"qid": q, "question": T[q]["question"], "answer": A[q]["answer"],
               "reference_answer": T[q]["reference_answer"], "correct": "", "faithful": ""} for q in picked]
             ).to_csv(results_dir / f"{pre}_manual_check.csv", index=False, encoding="utf-8-sig")

# failures: wrong citation or wrong refusal
fails = []
for q, t in T.items():
    a = A[q]
    bad_refusal = a["refused"] != (not t["answerable"])
    bad_cite = t["answerable"] and not a["refused"] and not (a["cited_chunks"] and set(a["cited_chunks"]) <= set(t["gold_chunks"]))
    if bad_refusal or bad_cite:
        fails.append({"qid": q, "question": t["question"], "gold_chunks": t["gold_chunks"],
                      "cited_chunks": a["cited_chunks"], "refused": a["refused"], "answer": a["answer"]})
pd.DataFrame(fails, columns=["qid", "question", "gold_chunks", "cited_chunks", "refused", "answer"]
             ).to_csv(results_dir / f"{pre}_failures.csv", index=False, encoding="utf-8-sig")
print(f"Saved 4 files in {results_dir} | failures: {len(fails)}")

tmp = DEMO / f"_pack_{pre}"
shutil.rmtree(tmp, ignore_errors=True); tmp.mkdir()
for suffix in ("_answers.json", "_summary.json", "_manual_check.csv", "_failures.csv"):
    shutil.copy(results_dir / f"{pre}{suffix}", tmp)
zip_path = shutil.make_archive(str(DEMO / pre), "zip", tmp)
shutil.rmtree(tmp)
print("Zipped:", zip_path)
if IS_COLAB:
    from google.colab import files as colab_files
    colab_files.download(zip_path)
else:
    print("On Kaggle: the zip is in the notebook output.")

## 9. Combine (run after all three `llm_gold_*_summary.json` are in `results/`)

In [ ]:
rows = []
for name in ("qwen3b", "qwen7b", "aya8b"):
    f = results_dir / f"llm_gold_{name}_summary.json"
    if f.exists():
        d = load_json(f)
        rows.append({k: d[k] for k in ("llm", "citation_acc", "cite_all_gold", "refusal_acc", "false_refusal",
                                      "correct_refusal", "avg_seconds", "peak_gpu_mb")})
    else:
        print("Missing:", f.name)
comb = pd.DataFrame(rows)
if len(comb):
    comb.to_csv(results_dir / "llm_gold_compare.csv", index=False)
    display(comb.set_index("llm"))
    print("With few questions, a gap of 1 or 2 is noise. Fill the manual_check files: correctness and faithfulness decide ties.")